In [0]:
import hashlib
from pyspark.sql import functions as F

# ============================================================
# Silver parser: executes a mapping on bronze data, in memory (dry run). Nothing is written here.
#   bronze rows -> parse steps (explode / zip_explode over the VARIANT) -> typed contract columns
#   -> source and lineage columns -> PK / FKs (md5)
# Conversions use try_* functions: a value that cannot be converted becomes null and is COUNTED
# in the dry-run report (it will be quarantined in the next phase), instead of failing the run.
# The Spark session time zone is set to UTC: timestamps are UTC instants and keys hash them as
# 'yyyy-MM-dd HH:mm:ss' (UTC), so FKs built anywhere match the PKs.
# Requires: common/_utils_config, common/_utils_reference_data,
#           silver/utils/_utils_silver_contracts, silver/utils/_utils_silver_mappings
# Cells: 1 constants/SQL expressions · 2 parse · 3 dry run report
# ============================================================

silver_session_timezone = "UTC"
dry_run_files = 2                    # latest bronze files used by a dry run
dry_run_sample_rows = 5
key_timestamp_format = "yyyy-MM-dd HH:mm:ss"
key_date_format = "yyyy-MM-dd"
raw_prefix = "__raw_"                # raw value (string) of every column mapped with 'from'


def use_utc_session():
    spark.conf.set("spark.sql.session.timeZone", silver_session_timezone)


def sql_string(value):
    """SQL string literal."""
    return "'" + str(value).replace("\\", "\\\\").replace("'", "\\'") + "'"


def sql_literal(value):
    """SQL literal for a string or a number."""
    return sql_string(value) if isinstance(value, str) else str(value)


def json_path(fields):
    """['attributes', 'values'] -> $['attributes']['values']"""
    return "$" + "".join(f"['{field}']" for field in fields)


def alias_column(alias, field=None):
    return f"`__alias_{alias}__{field}`" if field else f"`__alias_{alias}`"


def variant_base(alias, fields, known):
    """(variant column, json path) for a path; zip_explode aliases hold one variant per field."""
    if known[alias] is not None:
        return alias_column(alias, fields[0]), "$"
    base = "payload" if alias == root_alias else alias_column(alias)
    return base, json_path(fields)


def variant_array_sql(path, known):
    """Array of variants at a path; a single object is treated as a one-element list."""
    alias, fields = split_path(path)
    base, path_json = variant_base(alias, fields, known)
    p = sql_string(path_json)
    return (f"CASE WHEN try_variant_get({base}, {p}, 'array<variant>') IS NOT NULL "
            f"THEN try_variant_get({base}, {p}, 'array<variant>') "
            f"WHEN try_variant_get({base}, {p}, 'variant') IS NOT NULL "
            f"THEN array(try_variant_get({base}, {p}, 'variant')) END")


def from_value_sql(source, known, is_csv, target_type):
    """Value of a 'from' (location_code, csv column or variant path) as target_type."""
    if source == location_code_field:
        value = "try_element_at(location_codes, __position + 1)"
        return value if target_type == "string" else f"try_cast({value} AS {target_type})"
    if is_csv:
        return f"`{source}`" if target_type == "string" else f"try_cast(`{source}` AS {target_type})"
    alias, fields = split_path(source)
    base, path_json = variant_base(alias, fields, known)
    return f"try_variant_get({base}, {sql_string(path_json)}, '{target_type}')"


def constant_sql(value, column_type):
    if value is None:
        return f"CAST(NULL AS {column_type})"
    if isinstance(value, bool):
        return f"CAST({str(value).lower()} AS {column_type})"
    if isinstance(value, str):
        return f"CAST({sql_string(value)} AS {column_type})"
    return f"CAST({value} AS {column_type})"


def typed_value_sql(column_name, rule, column, known, is_csv):
    """SQL expression of a contract column, from its mapping rule (raw values are in __raw_<column>)."""
    column_type = column["type"]
    if "constant" in rule:
        return constant_sql(rule["constant"], column_type)

    raw = f"`{raw_prefix}{column_name}`"
    if "map_values" in rule:
        pairs = ", ".join(f"{sql_string(k)}, {sql_literal(v)}" for k, v in rule["map_values"].items())
        mapped = f"try_element_at(map({pairs}), {raw})"
        return mapped if column_type == "string" else f"try_cast({mapped} AS {column_type})"
    if column_type == "string":
        return raw
    if column_type in ("timestamp", "date"):
        timestamp = (f"try_to_timestamp({raw}, {sql_string(rule['format'])})" if "format" in rule
                     else f"try_to_timestamp({raw})")
        timezone = rule.get("timezone")
        if timezone and timezone not in timezone_modes:          # IANA zone: local wall time -> UTC
            timestamp = f"to_utc_timestamp({timestamp}, {sql_string(timezone)})"
        return timestamp if column_type == "timestamp" else f"to_date({timestamp})"

    value = from_value_sql(rule["from"], known, is_csv, column_type)
    if "multiply" in rule:
        value = f"try_cast(({value}) * {rule['multiply']} AS {column_type})"
    return value


def key_value_sql(column_name, column_type):
    if column_type == "timestamp":
        return f"date_format(`{column_name}`, '{key_timestamp_format}')"
    if column_type == "date":
        return f"date_format(`{column_name}`, '{key_date_format}')"
    return f"CAST(`{column_name}` AS STRING)"


def key_hash_sql(columns_with_types):
    """The ONLY key function of the platform: md5(concat_ws('|', key values))."""
    return "md5(concat_ws('|', " + ", ".join(key_value_sql(c, t) for c, t in columns_with_types) + "))"


def key_hash_python(values):
    """Same hash in Python (tests, reference data): values already formatted as strings."""
    return hashlib.md5("|".join(values).encode("utf-8")).hexdigest()

In [0]:
# ------------------------------------------------------------
# Parse: bronze DataFrame + mapping -> silver DataFrame (plus __raw_ columns for the report)
# ------------------------------------------------------------
def apply_parse_steps(df, mapping):
    """Applies explode / zip_explode steps. Returns (df, known aliases)."""
    known = {root_alias: None}
    for step in mapping.get("parse") or []:
        if "explode" in step:
            array_col = F.expr(variant_array_sql(step["explode"], known))
            target = f"__alias_{step['as']}"
            if "position" in step:
                df = df.select("*", F.posexplode(array_col).alias("__position", target))
            else:
                df = df.select("*", F.explode(array_col).alias(target))
            known[step["as"]] = None
        else:
            body = step["zip_explode"]
            alias, fields = body["as"], list(body["arrays"])
            arrays = {f"__zip_{alias}__{f}": F.expr(variant_array_sql(path, known)) for f, path in body["arrays"].items()}
            df = df.withColumns(arrays)
            df = df.select("*", F.posexplode(F.col(f"__zip_{alias}__{fields[0]}"))
                                  .alias("__zip_index", f"__alias_{alias}__{fields[0]}"))
            for field in fields[1:]:
                df = df.withColumn(f"__alias_{alias}__{field}",
                                   F.expr(f"try_element_at(`__zip_{alias}__{field}`, __zip_index + 1)"))
            df = df.drop(*arrays.keys(), "__zip_index")
            known[alias] = set(fields)
    return df, known


def parse_mapping(mapping, contracts, bronze_df, silver_run_id="dry_run"):
    """Silver columns (PK, FKs, contract columns, source, lineage) + __raw_ columns, in memory."""
    use_utc_session()
    contract = contracts[mapping["entity"]]
    source = mapping["source"]
    _, endpoint = find_bronze_endpoint(source["source_name"], source["endpoint"])
    is_csv = endpoint.get("response_format") == "csv"

    df, known = (bronze_df, {}) if is_csv else apply_parse_steps(bronze_df, mapping)
    rules = mapping["columns"]

    raw = {f"{raw_prefix}{c['name']}": F.expr(from_value_sql(rules[c["name"]]["from"], known, is_csv, "string"))
           for c in contract["columns"] if "from" in rules[c["name"]]}
    df = df.withColumns(raw)
    df = df.withColumns({c["name"]: F.expr(typed_value_sql(c["name"], rules[c["name"]], c, known, is_csv))
                         for c in contract["columns"]})

    record_id = (F.expr(from_value_sql(mapping["source_record_id"], known, is_csv, "string"))
                 if "source_record_id" in mapping else F.lit(None).cast("string"))
    df = df.withColumns({
        "source_name": F.lit(source["source_name"]),
        "source_endpoint": F.lit(source["endpoint"]),
        "source_record_id": record_id,
        "bronze_table": F.lit(bronze_table_for_mapping(mapping)),
        "bronze_run_id": F.col("run_id"),
        "silver_run_id": F.lit(silver_run_id),
        "processed_at": F.current_timestamp(),
    })

    table = build_silver_tables(contract)[0]
    types = {c["name"]: c["type"] for c in contract["columns"]}
    types["source_name"] = "string"
    keys = {table["primary_key"]["name"]: key_hash_sql([(c, types[c]) for c in table["primary_key"]["columns"]])}
    for fk in table["foreign_keys"]:
        keys[fk["name"]] = (f"md5({sql_string(fk['constant'])})" if fk["constant"]
                            else key_hash_sql([(c, types[c]) for c in fk["columns"]]))
    df = df.withColumns({name: F.expr(sql) for name, sql in keys.items()})

    silver_columns = [name for name, _ in table["columns"]]
    return df.select(*silver_columns, *raw.keys())

In [0]:
# ------------------------------------------------------------
# Dry run: latest bronze files -> parse -> report (nothing is written)
# ------------------------------------------------------------
def read_bronze_sample(mapping, files=dry_run_files):
    """Rows of the latest bronze files of the mapping's table. Returns (df, file paths)."""
    bronze = spark.table(bronze_table_for_mapping(mapping))
    latest = [r["source_file"] for r in (bronze.select("source_file", "chunk_start").distinct()
                                         .orderBy(F.desc("chunk_start")).limit(files).collect())]
    return bronze.filter(F.col("source_file").isin(latest)), latest


def build_dry_run_report(parsed_df, mapping, contracts):
    contract = contracts[mapping["entity"]]
    table = build_silver_tables(contract)[0]
    pk = table["primary_key"]["name"]
    rules = mapping["columns"]
    from_columns = [c["name"] for c in contract["columns"] if "from" in rules[c["name"]]]
    time_columns = [c["name"] for c in contract["columns"] if c["type"] in ("timestamp", "date")]

    aggregations = [F.count(F.lit(1)).alias("rows")]
    aggregations += [F.sum(F.col(c["name"]).isNull().cast("int")).alias(f"nulls__{c['name']}")
                     for c in contract["columns"]]
    aggregations += [F.sum((F.col(f"{raw_prefix}{c}").isNotNull() & F.col(c).isNull()).cast("int"))
                     .alias(f"failed__{c}") for c in from_columns]
    aggregations += [F.min(c).alias(f"min__{c}") for c in time_columns]
    aggregations += [F.max(c).alias(f"max__{c}") for c in time_columns]
    stats = parsed_df.agg(*aggregations).first().asDict()

    unmapped = {}
    for c in from_columns:
        if "map_values" in rules[c] and stats[f"failed__{c}"]:
            rows = (parsed_df.filter(F.col(f"{raw_prefix}{c}").isNotNull() & F.col(c).isNull())
                    .groupBy(f"{raw_prefix}{c}").count().orderBy(F.desc("count")).limit(10).collect())
            unmapped[c] = {r[0]: r["count"] for r in rows}

    duplicated_pks = parsed_df.groupBy(pk).count().filter(F.col("count") > 1).count()
    shown = [name for name, _ in table["columns"] if name not in erd_hidden_columns]
    return {
        "mapping": mapping["mapping"],
        "rows": stats["rows"],
        "nulls": {c["name"]: stats[f"nulls__{c['name']}"] or 0 for c in contract["columns"]},
        "failed_conversions": {c: stats[f"failed__{c}"] or 0 for c in from_columns},
        "unmapped_values": unmapped,
        "duplicated_pks": duplicated_pks,
        "time_range": {c: (stats[f"min__{c}"], stats[f"max__{c}"]) for c in time_columns},
        "sample": parsed_df.select(*shown).limit(dry_run_sample_rows),
    }


def dry_run_mapping(mapping, contracts, files=dry_run_files):
    """Executes a mapping on the latest bronze files. Returns (parsed DataFrame, report)."""
    bronze_df, source_files = read_bronze_sample(mapping, files)
    parsed = parse_mapping(mapping, contracts, bronze_df)
    report = build_dry_run_report(parsed, mapping, contracts)
    report["source_files"] = source_files
    return parsed, report


def print_dry_run_report(report):
    print(f"🧪 Dry run · {report['mapping']}")
    for path in report["source_files"]:
        print(f"   file: {path}")
    print(f"   rows: {report['rows']} · duplicated PKs: {report['duplicated_pks']}")
    for column, (low, high) in report["time_range"].items():
        print(f"   {column}: {low} -> {high} (UTC)")
    print("   column                       nulls   failed conversions")
    for column, nulls in report["nulls"].items():
        failed = report["failed_conversions"].get(column, "-")
        flag = " ⚠️" if (isinstance(failed, int) and failed) else ""
        print(f"   {column:<28} {nulls:>6}   {failed}{flag}")
    for column, values in report["unmapped_values"].items():
        print(f"   ⚠️ unmapped codes in {column}: {values}")
    display(report["sample"])